# Advanced Problems with Solutions: Lazy Iterables

This notebook extends the topic of **lazy iterables** with advanced, practical exercises.

A value is *lazy* when it is not computed until it is needed. In Python, laziness commonly appears in:

- generator functions;
- iterator classes;
- cached properties;
- streaming pipelines;
- infinite sequences;
- paginated APIs;
- file and CSV processing.

Best practices used throughout this notebook:

- avoid materializing large or infinite iterables;
- use `itertools.islice` to inspect finite parts of infinite streams;
- separate iterable objects from iterator objects when reusability matters;
- validate edge cases early;
- write tests that prove laziness by checking how much input was consumed.


## Setup


In [1]:
from __future__ import annotations

import csv
import heapq
import math
from collections import deque
from functools import wraps
from itertools import count, islice
from typing import Any, Callable, Generic, Iterable, Iterator, Sequence, TypeVar

T = TypeVar("T")
U = TypeVar("U")


## Problem 1 — Implement a reusable `lazy_property`

Create a descriptor called `lazy_property`.

Requirements:

1. It should compute the property value only when first requested.
2. It should cache the computed value on the instance.
3. Repeated access should return the cached value without recomputing.
4. It should preserve the wrapped function's name and docstring.
5. Create a helper function `invalidate_lazy(instance, *names)` that clears selected cached lazy properties.
6. Demonstrate the descriptor with a `Circle` class where changing `radius` invalidates `area` and `circumference`.

This problem connects lazy iterables with the broader idea of lazy evaluation: defer expensive work until needed.


In [2]:
class lazy_property:
    """Descriptor that computes a value once and caches it on the instance."""

    def __init__(self, func: Callable[[Any], Any]):
        wraps(func)(self)
        self.func = func
        self.cache_name = f"_lazy_{func.__name__}"

    def __get__(self, instance: Any, owner: type | None = None) -> Any:
        if instance is None:
            return self

        if self.cache_name not in instance.__dict__:
            instance.__dict__[self.cache_name] = self.func(instance)

        return instance.__dict__[self.cache_name]


def invalidate_lazy(instance: Any, *names: str) -> None:
    """Remove selected cached lazy-property values from an instance."""
    for name in names:
        instance.__dict__.pop(f"_lazy_{name}", None)


class Circle:
    def __init__(self, radius: float):
        self.radius = radius
        self.area_calculations = 0
        self.circumference_calculations = 0

    @property
    def radius(self) -> float:
        return self._radius

    @radius.setter
    def radius(self, value: float) -> None:
        if value < 0:
            raise ValueError("radius must not be negative")
        self._radius = value
        invalidate_lazy(self, "area", "circumference")

    @lazy_property
    def area(self) -> float:
        """Area of the circle, computed lazily."""
        self.area_calculations += 1
        return math.pi * self.radius ** 2

    @lazy_property
    def circumference(self) -> float:
        """Circumference of the circle, computed lazily."""
        self.circumference_calculations += 1
        return 2 * math.pi * self.radius


c = Circle(3)
print(c.__dict__)
print(c.area)
print(c.area)
print(c.__dict__)


{'_radius': 3, 'area_calculations': 0, 'circumference_calculations': 0}
28.274333882308138
28.274333882308138
{'_radius': 3, 'area_calculations': 1, 'circumference_calculations': 0, '_lazy_area': 28.274333882308138}


In [3]:
# Tests
c = Circle(2)

assert c.area_calculations == 0
assert round(c.area, 5) == round(math.pi * 4, 5)
assert c.area_calculations == 1

# Second access uses cache.
assert round(c.area, 5) == round(math.pi * 4, 5)
assert c.area_calculations == 1

# Different lazy property has its own cache.
assert round(c.circumference, 5) == round(4 * math.pi, 5)
assert c.circumference_calculations == 1
assert c.circumference_calculations == 1

# Radius mutation invalidates both lazy values.
c.radius = 5
assert round(c.area, 5) == round(25 * math.pi, 5)
assert c.area_calculations == 2
assert round(c.circumference, 5) == round(10 * math.pi, 5)
assert c.circumference_calculations == 2

assert Circle.area.__name__ == "area"
assert "computed lazily" in Circle.area.__doc__

print("Problem 1 tests passed.")


Problem 1 tests passed.


## Problem 2 — Build a reusable lazy iterable of squares

Implement `LazySquares(start=0, stop=None)`.

Requirements:

1. If `stop` is an integer, iterate over `start, start + 1, ..., stop - 1` and yield their squares.
2. If `stop is None`, produce an infinite stream of squares.
3. The object must be a reusable iterable: calling `iter(obj)` twice should create two independent iterators.
4. Validate that `start` and `stop` are integers when provided.
5. Avoid precomputing or storing all squares.

This problem highlights the difference between an **iterable** and an **iterator**. A reusable iterable returns a fresh iterator every time `__iter__` is called.


In [4]:
class LazySquares:
    """Reusable lazy iterable that yields square numbers."""

    def __init__(self, start: int = 0, stop: int | None = None):
        if not isinstance(start, int):
            raise TypeError("start must be an integer")
        if stop is not None and not isinstance(stop, int):
            raise TypeError("stop must be an integer or None")

        self.start = start
        self.stop = stop

    def __iter__(self) -> Iterator[int]:
        current = self.start
        while self.stop is None or current < self.stop:
            yield current ** 2
            current += 1

    def __repr__(self) -> str:
        return f"LazySquares(start={self.start}, stop={self.stop})"


finite_squares = LazySquares(3, 8)
list(finite_squares)


[9, 16, 25, 36, 49]

In [5]:
# Tests
squares = LazySquares(1, 5)
assert list(squares) == [1, 4, 9, 16]
assert list(squares) == [1, 4, 9, 16]  # Reusable: second pass still works.

infinite_squares = LazySquares(10)
assert list(islice(infinite_squares, 6)) == [100, 121, 144, 169, 196, 225]

iterator_a = iter(LazySquares(2, 6))
iterator_b = iter(LazySquares(2, 6))
assert next(iterator_a) == 4
assert next(iterator_a) == 9
assert next(iterator_b) == 4  # Independent iterator.

try:
    LazySquares(0, 3.5)
except TypeError:
    pass
else:
    raise AssertionError("Expected TypeError for non-integer stop")

print("Problem 2 tests passed.")


Problem 2 tests passed.


## Problem 3 — Infinite factorial stream without repeated full recalculation

Implement a generator function `factorials()` that yields:

```python
0!, 1!, 2!, 3!, 4!, ...
```

Requirements:

1. Produce an infinite stream.
2. Do not call `math.factorial` for every value.
3. Reuse the previous factorial to compute the next one.
4. Use `islice` to test finite prefixes safely.

The key recurrence is:

```python
(n + 1)! = n! * (n + 1)
```


In [6]:
def factorials() -> Iterator[int]:
    n = 0
    current = 1

    while True:
        yield current
        n += 1
        current *= n


list(islice(factorials(), 10))


[1, 1, 2, 6, 24, 120, 720, 5040, 40320, 362880]

In [7]:
# Tests
assert list(islice(factorials(), 10)) == [1, 1, 2, 6, 24, 120, 720, 5040, 40320, 362880]
assert list(islice(factorials(), 1)) == [1]
assert next(islice(factorials(), 20, None)) == math.factorial(20)

# The stream is lazy: we can ask for a tiny prefix of an infinite sequence.
first_three = list(islice(factorials(), 3))
assert first_three == [1, 1, 2]

print("Problem 3 tests passed.")


Problem 3 tests passed.


## Problem 4 — Safe utilities for consuming lazy iterables

Implement three utility functions:

1. `take(n, iterable)` — return the first `n` items as a list.
2. `nth(iterable, n, default=None)` — return the item at zero-based index `n`, or `default` if missing.
3. `consume(iterable, n=None)` — advance an iterator by `n` items, or consume it completely if `n is None`.

Requirements:

- Validate that `n` is not negative.
- Use `itertools.islice` where appropriate.
- These functions must work with infinite iterables.

These are common helper functions when testing lazy code.


In [8]:
def take(n: int, iterable: Iterable[T]) -> list[T]:
    if n < 0:
        raise ValueError("n must not be negative")
    return list(islice(iterable, n))


def nth(iterable: Iterable[T], n: int, default: T | None = None) -> T | None:
    if n < 0:
        raise ValueError("n must not be negative")
    return next(islice(iterable, n, None), default)


def consume(iterable: Iterable[T], n: int | None = None) -> None:
    iterator = iter(iterable)
    if n is None:
        deque(iterator, maxlen=0)
        return

    if n < 0:
        raise ValueError("n must not be negative")

    deque(islice(iterator, n), maxlen=0)


take(8, count(10, 3))


[10, 13, 16, 19, 22, 25, 28, 31]

In [9]:
# Tests
assert take(5, count(1)) == [1, 2, 3, 4, 5]
assert nth(count(10, 10), 4) == 50
assert nth(["a", "b"], 10, default="missing") == "missing"

iterator = iter([1, 2, 3, 4, 5])
consume(iterator, 2)
assert list(iterator) == [3, 4, 5]

iterator = iter([1, 2, 3])
consume(iterator)
assert list(iterator) == []

for bad_call in [
    lambda: take(-1, []),
    lambda: nth([], -1),
    lambda: consume([], -1),
]:
    try:
        bad_call()
    except ValueError:
        pass
    else:
        raise AssertionError("Expected ValueError for negative n")

print("Problem 4 tests passed.")


Problem 4 tests passed.


## Problem 5 — Build and test a lazy map/filter pipeline

Implement `lazy_map` and `lazy_filter`.

Requirements:

1. Both functions must return iterators.
2. They must process items only when requested.
3. Use tests that prove only the minimum required source values are consumed.
4. Do not use list comprehensions inside the implementations.

This exercise is about *observable laziness*: the test should reveal that work happens only as downstream consumers ask for values.


In [10]:
def lazy_map(func: Callable[[T], U], iterable: Iterable[T]) -> Iterator[U]:
    for item in iterable:
        yield func(item)


def lazy_filter(predicate: Callable[[T], bool], iterable: Iterable[T]) -> Iterator[T]:
    for item in iterable:
        if predicate(item):
            yield item


log: list[tuple[str, int]] = []


def noisy_numbers(limit: int) -> Iterator[int]:
    for number in range(limit):
        log.append(("source", number))
        yield number


pipeline = lazy_map(lambda x: x * x, lazy_filter(lambda x: x % 2 == 0, noisy_numbers(10)))
print(next(pipeline))
print(log)
print(next(pipeline))
print(log)


0
[('source', 0)]
4
[('source', 0), ('source', 1), ('source', 2)]


In [11]:
# Tests
log = []
pipeline = lazy_map(lambda x: x * x, lazy_filter(lambda x: x % 2 == 0, noisy_numbers(10)))

assert log == []
assert next(pipeline) == 0
assert log == [("source", 0)]

assert next(pipeline) == 4
assert log == [("source", 0), ("source", 1), ("source", 2)]

assert list(pipeline) == [16, 36, 64]
assert log == [("source", i) for i in range(10)]

print("Problem 5 tests passed.")


Problem 5 tests passed.


## Problem 6 — Lazy sliding windows

Implement `sliding_windows(iterable, size)`.

It should lazily produce overlapping windows of length `size`.

Example:

```python
list(sliding_windows([1, 2, 3, 4], 3))
# [(1, 2, 3), (2, 3, 4)]
```

Requirements:

1. Return an iterator.
2. Do not materialize the entire input.
3. Work with infinite iterables.
4. If there are not enough values for one full window, yield nothing.
5. Validate that `size` is positive.


In [12]:
def sliding_windows(iterable: Iterable[T], size: int) -> Iterator[tuple[T, ...]]:
    if size <= 0:
        raise ValueError("size must be positive")

    iterator = iter(iterable)
    window = deque(islice(iterator, size), maxlen=size)

    if len(window) < size:
        return

    yield tuple(window)

    for item in iterator:
        window.append(item)
        yield tuple(window)


list(sliding_windows([1, 2, 3, 4, 5], 3))


[(1, 2, 3), (2, 3, 4), (3, 4, 5)]

In [13]:
# Tests
assert list(sliding_windows([1, 2, 3, 4, 5], 3)) == [
    (1, 2, 3),
    (2, 3, 4),
    (3, 4, 5),
]

assert list(sliding_windows([1, 2], 3)) == []
assert list(sliding_windows("ABCD", 2)) == [("A", "B"), ("B", "C"), ("C", "D")]
assert take(4, sliding_windows(count(1), 4)) == [
    (1, 2, 3, 4),
    (2, 3, 4, 5),
    (3, 4, 5, 6),
    (4, 5, 6, 7),
]

try:
    list(sliding_windows([1, 2, 3], 0))
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for non-positive size")

print("Problem 6 tests passed.")


Problem 6 tests passed.


## Problem 7 — Lazy chunking with optional incomplete final chunk

Implement `chunked(iterable, size, *, keep_incomplete=True)`.

Requirements:

1. Yield tuples of length `size`.
2. Do not materialize the full source iterable.
3. If `keep_incomplete` is `True`, yield the final short chunk.
4. If `keep_incomplete` is `False`, drop the final short chunk.
5. Work with infinite iterables.
6. Validate that `size` is positive.

Chunking is useful in streaming systems: you can batch records while still processing lazily.


In [14]:
def chunked(
    iterable: Iterable[T],
    size: int,
    *,
    keep_incomplete: bool = True,
) -> Iterator[tuple[T, ...]]:
    if size <= 0:
        raise ValueError("size must be positive")

    iterator = iter(iterable)

    while True:
        chunk = tuple(islice(iterator, size))
        if not chunk:
            return
        if len(chunk) == size or keep_incomplete:
            yield chunk


list(chunked(range(10), 4))


[(0, 1, 2, 3), (4, 5, 6, 7), (8, 9)]

In [15]:
# Tests
assert list(chunked(range(10), 4)) == [(0, 1, 2, 3), (4, 5, 6, 7), (8, 9)]
assert list(chunked(range(10), 4, keep_incomplete=False)) == [(0, 1, 2, 3), (4, 5, 6, 7)]
assert list(chunked([], 3)) == []
assert take(3, chunked(count(1), 5)) == [
    (1, 2, 3, 4, 5),
    (6, 7, 8, 9, 10),
    (11, 12, 13, 14, 15),
]

try:
    list(chunked([1, 2, 3], -2))
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for non-positive size")

print("Problem 7 tests passed.")


Problem 7 tests passed.


## Problem 8 — Memoized lazy iterable for one-shot sources

A generator is usually **one-shot**: once consumed, its values are gone.

Build a class `MemoizedIterable` that wraps a one-shot source and makes it replayable by caching values as they are requested.

Requirements:

1. The source should not be fully consumed at construction time.
2. Values should be cached only as they are requested.
3. Multiple iterators over the same `MemoizedIterable` should share the cache.
4. If one iterator has already forced values into the cache, another iterator should reuse those cached values without consuming the source again.
5. Once the source is exhausted, future iterations should replay the cache and then stop.

This is a powerful pattern, but it has a tradeoff: cached values use memory.


In [16]:
class MemoizedIterable(Generic[T]):
    """Replayable lazy wrapper around a one-shot source."""

    def __init__(self, source: Iterable[T]):
        self._source = iter(source)
        self._cache: list[T] = []
        self._exhausted = False

    def __iter__(self) -> Iterator[T]:
        index = 0

        while True:
            if index < len(self._cache):
                yield self._cache[index]
                index += 1
                continue

            if self._exhausted:
                return

            try:
                value = next(self._source)
            except StopIteration:
                self._exhausted = True
                return

            self._cache.append(value)
            yield value
            index += 1

    @property
    def cached_count(self) -> int:
        return len(self._cache)


source_log: list[int] = []


def source_numbers() -> Iterator[int]:
    for number in [10, 20, 30, 40]:
        source_log.append(number)
        yield number


memoized = MemoizedIterable(source_numbers())
first = iter(memoized)
print(next(first))
print(next(first))
print(source_log)
print(list(memoized))
print(source_log)


10
20
[10, 20]
[10, 20, 30, 40]
[10, 20, 30, 40]


In [17]:
# Tests
source_log = []
memoized = MemoizedIterable(source_numbers())

it1 = iter(memoized)
assert next(it1) == 10
assert next(it1) == 20
assert source_log == [10, 20]
assert memoized.cached_count == 2

# The second iterator reuses cached values first.
it2 = iter(memoized)
assert next(it2) == 10
assert next(it2) == 20
assert source_log == [10, 20]

# Continuing it2 forces exactly one more source value.
assert next(it2) == 30
assert source_log == [10, 20, 30]
assert memoized.cached_count == 3

# Exhaust everything.
assert list(memoized) == [10, 20, 30, 40]
assert source_log == [10, 20, 30, 40]
assert list(memoized) == [10, 20, 30, 40]
assert source_log == [10, 20, 30, 40]

print("Problem 8 tests passed.")


Problem 8 tests passed.


## Problem 9 — Lazy pagination over an API-like data source

Many APIs return data in pages. A bad implementation fetches all pages immediately. A lazy implementation fetches each page only when more records are needed.

Implement `iter_paginated_records(fetch_page, start_token=None)`.

Assume `fetch_page(token)` returns:

```python
(records, next_token)
```

where `next_token` is `None` when there are no more pages.

Requirements:

1. Fetch the first page only when iteration begins.
2. Fetch subsequent pages only after the current page is exhausted.
3. Yield individual records, not pages.
4. Stop cleanly when `next_token is None`.
5. Write tests that count API calls.


In [18]:
class FakePager:
    def __init__(self, pages: dict[str | None, tuple[list[T], str | None]]):
        self.pages = pages
        self.calls: list[str | None] = []

    def fetch(self, token: str | None) -> tuple[list[T], str | None]:
        self.calls.append(token)
        return self.pages.get(token, ([], None))


def iter_paginated_records(
    fetch_page: Callable[[str | None], tuple[Sequence[T], str | None]],
    start_token: str | None = None,
) -> Iterator[T]:
    token = start_token

    while True:
        records, next_token = fetch_page(token)

        for record in records:
            yield record

        if next_token is None:
            return

        token = next_token


pages = {
    None: (["A", "B"], "page-2"),
    "page-2": (["C", "D"], "page-3"),
    "page-3": (["E"], None),
}

pager = FakePager(pages)
records = iter_paginated_records(pager.fetch)
print(take(3, records))
print(pager.calls)


['A', 'B', 'C']
[None, 'page-2']


In [19]:
# Tests
pages = {
    None: (["A", "B"], "page-2"),
    "page-2": (["C", "D"], "page-3"),
    "page-3": (["E"], None),
}

pager = FakePager(pages)
records = iter_paginated_records(pager.fetch)

# No fetch happens before iteration.
assert pager.calls == []

assert take(1, records) == ["A"]
assert pager.calls == [None]

# Need records from page 1 and page 2.
assert take(3, records) == ["B", "C", "D"]
assert pager.calls == [None, "page-2"]

# Remaining records force page 3.
assert list(records) == ["E"]
assert pager.calls == [None, "page-2", "page-3"]

empty_pager = FakePager({None: ([], None)})
assert list(iter_paginated_records(empty_pager.fetch)) == []
assert empty_pager.calls == [None]

print("Problem 9 tests passed.")


Problem 9 tests passed.


## Problem 10 — Lazy CSV row processing

Implement two functions:

1. `iter_csv_dicts(lines)` — lazily parse CSV lines into dictionaries.
2. `filter_rows(rows, predicate)` — lazily yield only rows matching a predicate.

Requirements:

1. Do not read all CSV rows into memory.
2. Work with any iterable of strings, including a generator.
3. Demonstrate that only enough lines are consumed to produce the first matching row.
4. Use Python's `csv.DictReader`.

This is a realistic lazy iterable pattern: process files or network streams one row at a time.


In [20]:
def iter_csv_dicts(lines: Iterable[str]) -> Iterator[dict[str, str]]:
    reader = csv.DictReader(lines)
    yield from reader


def filter_rows(
    rows: Iterable[dict[str, str]],
    predicate: Callable[[dict[str, str]], bool],
) -> Iterator[dict[str, str]]:
    for row in rows:
        if predicate(row):
            yield row


line_log: list[str] = []


def csv_lines() -> Iterator[str]:
    for line in [
        "name,score\n",
        "Ada,81\n",
        "Linus,88\n",
        "Grace,95\n",
        "Guido,99\n",
    ]:
        line_log.append(line.strip())
        yield line


rows = iter_csv_dicts(csv_lines())
high_scores = filter_rows(rows, lambda row: int(row["score"]) >= 90)
print(next(high_scores))
print(line_log)


{'name': 'Grace', 'score': '95'}
['name,score', 'Ada,81', 'Linus,88', 'Grace,95']


In [21]:
# Tests
line_log = []
rows = iter_csv_dicts(csv_lines())
high_scores = filter_rows(rows, lambda row: int(row["score"]) >= 90)

assert next(high_scores) == {"name": "Grace", "score": "95"}

# Header, Ada, Linus, and Grace were consumed. Guido was not needed yet.
assert line_log == ["name,score", "Ada,81", "Linus,88", "Grace,95"]

assert next(high_scores) == {"name": "Guido", "score": "99"}
assert line_log == ["name,score", "Ada,81", "Linus,88", "Grace,95", "Guido,99"]

assert list(high_scores) == []

print("Problem 10 tests passed.")


Problem 10 tests passed.


## Problem 11 — Infinite lazy prime generator

Implement `primes()`.

Requirements:

1. Yield prime numbers forever.
2. Use previously found primes to test new candidates.
3. Stop divisibility checks once the known prime exceeds `sqrt(candidate)`.
4. Use `islice` to test finite prefixes.
5. Avoid precomputing an arbitrary upper bound.

This is a classic example of an infinite lazy iterable.


In [22]:
def primes() -> Iterator[int]:
    found: list[int] = []
    candidate = 2

    while True:
        limit = int(candidate ** 0.5)
        is_prime = True

        for prime in found:
            if prime > limit:
                break
            if candidate % prime == 0:
                is_prime = False
                break

        if is_prime:
            found.append(candidate)
            yield candidate

        candidate += 1 if candidate == 2 else 2


list(islice(primes(), 15))


[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]

In [23]:
# Tests
assert list(islice(primes(), 15)) == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]
assert nth(primes(), 0) == 2
assert nth(primes(), 9) == 29
assert nth(primes(), 24) == 97

# Can combine with other lazy utilities.
prime_windows = take(4, sliding_windows(primes(), 3))
assert prime_windows == [
    (2, 3, 5),
    (3, 5, 7),
    (5, 7, 11),
    (7, 11, 13),
]

print("Problem 11 tests passed.")


Problem 11 tests passed.


## Problem 12 — Lazy merge of sorted finite or infinite streams

Implement `merge_sorted_lazy(*iterables)`.

It should merge multiple sorted input iterables into one sorted output stream.

Requirements:

1. Return an iterator.
2. Do not materialize the input streams.
3. Work when some inputs are infinite.
4. Work when some inputs are empty.
5. Preserve duplicates.
6. Use a heap for efficiency.

This is a useful streaming pattern for logs, events, timestamps, or sorted numeric sequences.


In [24]:
def merge_sorted_lazy(*iterables: Iterable[T]) -> Iterator[T]:
    heap: list[tuple[T, int, Iterator[T]]] = []

    for source_id, iterable in enumerate(iterables):
        iterator = iter(iterable)
        try:
            first_value = next(iterator)
        except StopIteration:
            continue
        heapq.heappush(heap, (first_value, source_id, iterator))

    while heap:
        value, source_id, iterator = heapq.heappop(heap)
        yield value

        try:
            next_value = next(iterator)
        except StopIteration:
            continue

        heapq.heappush(heap, (next_value, source_id, iterator))


def arithmetic_stream(start: int, step: int) -> Iterator[int]:
    value = start
    while True:
        yield value
        value += step


merged = merge_sorted_lazy(arithmetic_stream(0, 2), arithmetic_stream(1, 2))
take(12, merged)


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

In [25]:
# Tests
assert take(12, merge_sorted_lazy(arithmetic_stream(0, 2), arithmetic_stream(1, 2))) == list(range(12))

assert list(merge_sorted_lazy([1, 4, 10], [2, 2, 3], [], [5, 8])) == [1, 2, 2, 3, 4, 5, 8, 10]

multiples_of_3 = arithmetic_stream(0, 3)
multiples_of_5 = arithmetic_stream(0, 5)
assert take(10, merge_sorted_lazy(multiples_of_3, multiples_of_5)) == [0, 0, 3, 5, 6, 9, 10, 12, 15, 15]

assert list(merge_sorted_lazy([], [])) == []

print("Problem 12 tests passed.")


Problem 12 tests passed.


## Final checklist: lazy iterable best practices

Use this checklist when designing lazy code:

- Prefer generator functions for simple lazy iterators.
- Use iterable classes when you need reusable iteration.
- Use iterator classes when you need explicit state and protocol control.
- Never call `list()` on an infinite iterable.
- Use `islice` to safely inspect finite prefixes.
- Test laziness by counting source consumption.
- Avoid hidden full materialization in helper functions.
- Be careful with caching: it improves replayability but uses memory.
- Separate data fetching from data processing when streaming.
- Validate edge cases such as empty inputs, negative sizes, and invalid tokens.
